###  SISTEMA DE QUALIDADE DE DADOS — 6 PILARES
Pilares avaliados:
1. Completude    — valores ausentes / nulos
2. Consistência  — duplicatas, conflitos de tipo
3. Conformidade  — formatos esperados (datas, padrões)
4. Integridade   — relacionamentos e valores inválidos
5. Precisão      — outliers e valores suspeitos
6. Atualidade    — quão recentes são os registros

In [63]:
import sys
import os
import re
import warnings
from datetime import datetime, date
 
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from io import BytesIO
 
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, HRFlowable, Image
)
from reportlab.graphics.shapes import Drawing, Rect, String
from reportlab.graphics import renderPDF
 
warnings.filterwarnings("ignore")

In [64]:
#  Constantes e configurações globais
CONFIG = {
    # Colunas que NUNCA devem ser nulas (lista de nomes exatos)
    "colunas_obrigatorias": [],
 
    # Colunas que devem ser únicas (ex: CPF, ID)
    "colunas_unicas": [],
 
    # Formatos de data aceitos (o script testa todos automaticamente)
    "formatos_data": ["%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y", "%Y/%m/%d", "%d.%m.%Y"],
 
    # Colunas que são datas (deixe vazio para detecção automática)
    "colunas_data": [],
 
    # Número de dias: registros mais antigos que isso geram alerta de atualidade
    "atualidade_limite_dias": 365,
 
    # Limiar de outlier via IQR (padrão 1.5)
    "outlier_iqr_fator": 1.5,
 
    # Porcentagem mínima aceitável de completude por coluna (0–100)
    "completude_minima_pct": 80.0,
}
 
 
# Entrada e saída
ARQUIVO_CSV = r"../dataset/Cleaned_Laptop_data.csv" 
ARQUIVO_PDF = None

#  Paleta de cores
COR_PRIMARIA   = colors.HexColor("#1B3A6B")   # azul escuro
COR_SECUNDARIA = colors.HexColor("#2E86C1")   # azul médio
COR_DESTAQUE   = colors.HexColor("#F39C12")   # âmbar
COR_SUCESSO    = colors.HexColor("#27AE60")   # verde
COR_ALERTA     = colors.HexColor("#E74C3C")   # vermelho
COR_NEUTRO     = colors.HexColor("#ECF0F1")   # cinza claro
COR_TEXTO      = colors.HexColor("#2C3E50")   # quase preto

## 1. LEITURA E PRÉ-PROCESSAMENTO

In [65]:
def carregar_csv(caminho: str) -> pd.DataFrame:
    """Tenta ler o CSV com diferentes encodings e separadores."""
    encodings = ["utf-8", "latin-1", "cp1252", "utf-8-sig"]
    separadores = [",", ";", "\t", "|"]
    for enc in encodings:
        for sep in separadores:
            try:
                df = pd.read_csv(caminho, encoding=enc, sep=sep, low_memory=False)
                if df.shape[1] > 1:
                    return df
            except Exception:
                continue
    raise ValueError(f"Não foi possível ler o arquivo: {caminho}")
 
 
def detectar_colunas_data(df: pd.DataFrame) -> list:
    """Detecta automaticamente colunas que parecem conter datas."""
    candidatas = []
    palavras_chave = ["data", "date", "dt_", "_dt", "vencimento", "nascimento",
                      "criacao", "criação", "abertura", "fechamento", "prazo"]
    for col in df.columns:
        col_lower = col.lower()
        if any(k in col_lower for k in palavras_chave):
            candidatas.append(col)
            continue
        # Testa amostra da coluna
        amostra = df[col].dropna().astype(str).head(20)
        for fmt in CONFIG["formatos_data"]:
            try:
                pd.to_datetime(amostra, format=fmt, errors="raise")
                candidatas.append(col)
                break
            except Exception:
                continue
    return list(set(candidatas))
 
 
def tentar_parse_data(serie: pd.Series) -> pd.Series:
    """Tenta converter uma série para datetime usando os formatos configurados."""
    for fmt in CONFIG["formatos_data"]:
        try:
            return pd.to_datetime(serie, format=fmt, errors="coerce")
        except Exception:
            continue
    return pd.to_datetime(serie, infer_datetime_format=True, errors="coerce")

## 2. AVALIAÇÃO DOS 6 PILARES

### SEÇÃO: COMPLETUDE

In [66]:
def avaliar_completude(df: pd.DataFrame) -> dict:
    total_celulas = df.shape[0] * df.shape[1]
    nulos = df.isnull().sum()
    pct_nulo = (nulos / len(df) * 100).round(2)
    pct_preenchido = (100 - pct_nulo).round(2)
 
    # Colunas que ficam abaixo do mínimo configurado
    problemas = pct_preenchido[pct_preenchido < CONFIG["completude_minima_pct"]].to_dict()
 
    # Score: média ponderada de preenchimento
    score = float(pct_preenchido.mean())
 
    detalhe = pd.DataFrame({
        "Coluna": nulos.index,
        "Nulos": nulos.values,
        "% Preenchido": pct_preenchido.values
    }).sort_values("% Preenchido")
 
    # Alertas para colunas obrigatórias
    alertas_obrig = []
    for col in CONFIG["colunas_obrigatorias"]:
        if col in df.columns and df[col].isnull().any():
            n = int(df[col].isnull().sum())
            alertas_obrig.append(f"'{col}': {n} nulo(s) em coluna obrigatória")
 
    return {
        "score": round(score, 2),
        "total_nulos": int(nulos.sum()),
        "total_celulas": total_celulas,
        "detalhe": detalhe,
        "colunas_criticas": problemas,
        "alertas_obrigatorias": alertas_obrig,
    }

### SEÇÃO: CONSISTÊNCIA

In [67]:
def avaliar_consistencia(df: pd.DataFrame) -> dict:
    # Duplicatas completas
    dup_completas = int(df.duplicated().sum())
    pct_dup = round(dup_completas / len(df) * 100, 2)
 
    # Duplicatas por colunas únicas configuradas
    dup_unicas = {}
    for col in CONFIG["colunas_unicas"]:
        if col in df.columns:
            n = int(df[col].dropna().duplicated().sum())
            if n > 0:
                dup_unicas[col] = n
 
    # Inconsistências de tipo: colunas numéricas com strings misturadas
    tipo_misto = {}
    for col in df.select_dtypes(include="object").columns:
        amostra = df[col].dropna().head(500)
        n_num = amostra.apply(lambda x: str(x).replace(",", ".").replace("-", "")
                               .replace(" ", "").isnumeric()).sum()
        if 0 < n_num < len(amostra) * 0.9 and n_num > len(amostra) * 0.1:
            tipo_misto[col] = {"numerico": int(n_num), "texto": int(len(amostra) - n_num)}
 
    penalidade = (pct_dup / 100) * 50 + min(len(dup_unicas) * 10, 30) + min(len(tipo_misto) * 5, 20)
    score = max(0, 100 - penalidade)
 
    return {
        "score": round(score, 2),
        "duplicatas_completas": dup_completas,
        "pct_duplicatas": pct_dup,
        "duplicatas_colunas_unicas": dup_unicas,
        "tipo_misto": tipo_misto,
    }

### SEÇÃO: CONFORMIDADE

In [68]:
def avaliar_conformidade(df: pd.DataFrame, colunas_data: list) -> dict:
    problemas_data = {}
    datas_invalidas_total = 0
 
    for col in colunas_data:
        if col not in df.columns:
            continue
        serie = df[col].dropna().astype(str)
        parsed = tentar_parse_data(serie)
        invalidas = int(parsed.isnull().sum())
        if invalidas > 0:
            problemas_data[col] = invalidas
            datas_invalidas_total += invalidas
 
    total_datas = sum(df[c].notna().sum() for c in colunas_data if c in df.columns)
    pct_invalido = (datas_invalidas_total / total_datas * 100) if total_datas > 0 else 0
    score = max(0, 100 - pct_invalido)
 
    return {
        "score": round(score, 2),
        "colunas_data_analisadas": colunas_data,
        "datas_invalidas": problemas_data,
        "total_datas_invalidas": datas_invalidas_total,
        "total_datas": int(total_datas),
    }

### SEÇÃO: INTEGRIDADE

In [69]:
def avaliar_integridade(df: pd.DataFrame) -> dict:
    problemas = {}
 
    # Verifica valores negativos em colunas que deveriam ser positivas
    colunas_num = df.select_dtypes(include=[np.number]).columns
    palavras_positivas = ["valor", "preco", "preço", "quantidade", "qtd", "qtde",
                          "total", "age", "idade", "salario", "salário"]
    negativos = {}
    for col in colunas_num:
        if any(p in col.lower() for p in palavras_positivas):
            n_neg = int((df[col] < 0).sum())
            if n_neg > 0:
                negativos[col] = n_neg
 
    # Verifica strings vazias (whitespace only)
    strings_vazias = {}
    for col in df.select_dtypes(include="object").columns:
        n = int((df[col].astype(str).str.strip() == "").sum())
        if n > 0:
            strings_vazias[col] = n
 
    total_problemas = sum(negativos.values()) + sum(strings_vazias.values())
    total_registros = len(df)
    penalidade = min((total_problemas / total_registros) * 100, 100) if total_registros > 0 else 0
    score = max(0, 100 - penalidade)
 
    return {
        "score": round(score, 2),
        "valores_negativos_suspeitos": negativos,
        "strings_vazias": strings_vazias,
        "total_problemas": total_problemas,
    }


### SEÇÃO: PRECISÃO

In [70]:
def avaliar_precisao(df: pd.DataFrame) -> dict:
    outliers = {}
    fator = CONFIG["outlier_iqr_fator"]
    colunas_num = df.select_dtypes(include=[np.number]).columns
 
    for col in colunas_num:
        serie = df[col].dropna()
        if len(serie) < 10:
            continue
        Q1 = serie.quantile(0.25)
        Q3 = serie.quantile(0.75)
        IQR = Q3 - Q1
        if IQR == 0:
            continue
        limite_inf = Q1 - fator * IQR
        limite_sup = Q3 + fator * IQR
        n_out = int(((serie < limite_inf) | (serie > limite_sup)).sum())
        if n_out > 0:
            outliers[col] = {
                "outliers": n_out,
                "pct": round(n_out / len(serie) * 100, 2),
                "limite_inf": round(float(limite_inf), 4),
                "limite_sup": round(float(limite_sup), 4),
                "min": round(float(serie.min()), 4),
                "max": round(float(serie.max()), 4),
            }
 
    total_out = sum(v["outliers"] for v in outliers.values())
    total_vals = sum(df[c].notna().sum() for c in colunas_num)
    pct_out = (total_out / total_vals * 100) if total_vals > 0 else 0
    score = max(0, 100 - pct_out * 2)
 
    return {
        "score": round(score, 2),
        "outliers_por_coluna": outliers,
        "total_outliers": total_out,
        "pct_outliers": round(pct_out, 2),
    }

### SEÇÃO: ATUALIDADE

In [71]:
def avaliar_atualidade(df: pd.DataFrame, colunas_data: list) -> dict:
    hoje = pd.Timestamp.today()
    limite_dias = CONFIG["atualidade_limite_dias"]
    analise = {}
 
    for col in colunas_data:
        if col not in df.columns:
            continue
        parsed = tentar_parse_data(df[col].dropna().astype(str))
        parsed = parsed.dropna()
        if len(parsed) == 0:
            continue
        diff = (hoje - parsed).dt.days
        desatualizados = int((diff > limite_dias).sum())
        media_dias = float(diff.mean())
        mais_recente = parsed.max()
        mais_antigo = parsed.min()
        analise[col] = {
            "registros_analisados": len(parsed),
            "desatualizados": desatualizados,
            "pct_desatualizados": round(desatualizados / len(parsed) * 100, 2),
            "media_dias_atraso": round(media_dias, 1),
            "data_mais_recente": str(mais_recente.date()),
            "data_mais_antiga": str(mais_antigo.date()),
        }
 
    if analise:
        media_pct = np.mean([v["pct_desatualizados"] for v in analise.values()])
        score = max(0, 100 - media_pct)
    else:
        score = 100.0
 
    return {
        "score": round(score, 2),
        "analise_por_coluna": analise,
        "limite_dias": limite_dias,
    }

## 3. GERAÇÃO DE GRÁFICOS

In [72]:
def grafico_radar(scores: dict) -> BytesIO:
    """Gráfico radar com os 6 pilares."""
    labels = list(scores.keys())
    valores = list(scores.values())
    N = len(labels)
 
    angulos = [n / float(N) * 2 * np.pi for n in range(N)]
    angulos += angulos[:1]
    valores_plot = valores + valores[:1]
 
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("#F0F4F8")
 
    # Aneis de fundo por faixa de qualidade
    for r, cor, alpha in [(100, "#E74C3C", 0.04), (80, "#F39C12", 0.05), (60, "#27AE60", 0.06)]:
        ax.fill(angulos, [r] * len(angulos), color=cor, alpha=alpha)
 
    ax.plot(angulos, valores_plot, "o-", linewidth=2.5, color="#2E86C1", zorder=3)
    ax.fill(angulos, valores_plot, alpha=0.20, color="#2E86C1", zorder=2)
 
    # Pontos com valor anotado
    for ang, val in zip(angulos[:-1], valores):
        ax.plot(ang, val, "o", markersize=8, color="#1B3A6B", zorder=4)
        offset = 8 if val < 90 else -8
        ax.annotate(f"{val:.0f}", xy=(ang, val), xytext=(ang, val + offset),
                    ha="center", va="center", fontsize=8, fontweight="bold",
                    color="#1B3A6B", zorder=5)
 
    ax.set_xticks(angulos[:-1])
    ax.set_xticklabels(labels, size=11, fontweight="bold", color="#2C3E50")
    ax.set_ylim(0, 110)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(["20", "40", "60", "80", "100"], size=8, color="#95A5A6")
    ax.grid(color="#BDC3C7", linestyle="--", linewidth=0.6, alpha=0.8)
    ax.spines["polar"].set_color("#BDC3C7")
    ax.tick_params(pad=14)
 
    plt.tight_layout(pad=2.5)
    buf = BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    buf.seek(0)
    return buf
 
 
def grafico_barras_completude(detalhe: pd.DataFrame) -> BytesIO:
    """Barras horizontais com % de preenchimento por coluna (top 15 piores)."""
    piores = detalhe.head(15)
    fig, ax = plt.subplots(figsize=(7, max(3, len(piores) * 0.45)))
    fig.patch.set_facecolor("white")
 
    cores = ["#E74C3C" if v < CONFIG["completude_minima_pct"] else "#27AE60"
             for v in piores["% Preenchido"]]
    bars = ax.barh(piores["Coluna"], piores["% Preenchido"], color=cores, height=0.6)
 
    for bar, val in zip(bars, piores["% Preenchido"]):
        ax.text(min(val + 1, 98), bar.get_y() + bar.get_height() / 2,
                f"{val:.1f}%", va="center", ha="left", fontsize=8, color="#2C3E50")
 
    ax.axvline(x=CONFIG["completude_minima_pct"], color="#F39C12",
               linestyle="--", linewidth=1.5, label=f"Mínimo ({CONFIG['completude_minima_pct']}%)")
    ax.set_xlim(0, 105)
    ax.set_xlabel("% Preenchido", fontsize=9)
    ax.set_title("Completude por Coluna (piores)", fontsize=10, fontweight="bold", color="#1B3A6B")
    ax.legend(fontsize=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.set_facecolor("#F8F9FA")
    plt.tight_layout()
 
    buf = BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    buf.seek(0)
    return buf
 
 
def grafico_score_geral(score_geral: float) -> BytesIO:
    """Gauge semicircular para o score geral."""
    import matplotlib.patches as mpatches
    from matplotlib.patches import Wedge
 
    fig, ax = plt.subplots(figsize=(5, 3.2))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-0.5, 1.35)
    ax.set_aspect("equal")
    ax.axis("off")
 
    # Faixas: 180°=esquerda(Crítico) → 0°=direita(Bom)
    # score 0   → 180° (extrema esquerda)
    # score 100 → 0°   (extrema direita)
    faixas = [
        (120, 180, "#E74C3C", 0.38),   # Crítico  — esquerda
        (60,  120, "#F39C12", 0.38),   # Moderado — centro
        (0,    60, "#27AE60", 0.38),   # Bom      — direita
    ]
    for theta1, theta2, cor, alpha in faixas:
        wedge = Wedge((0, 0), 1.0, theta1, theta2, width=0.35,
                      facecolor=cor, alpha=alpha, edgecolor="white", linewidth=2)
        ax.add_patch(wedge)
 
    # Agulha: score 0→180°, score 100→0°
    angulo_graus = 180 - (score_geral * 1.8)
    angulo_rad = np.radians(angulo_graus)
    needle_x = 0.70 * np.cos(angulo_rad)
    needle_y = 0.70 * np.sin(angulo_rad)
    ax.annotate("", xy=(needle_x, needle_y), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color="#1B3A6B",
                                lw=2.5, mutation_scale=14))
 
    # Circulo central
    centro = plt.Circle((0, 0), 0.08, color="#1B3A6B", zorder=5)
    ax.add_patch(centro)
 
    # Score no centro
    ax.text(0, -0.32, f"{score_geral:.1f}", ha="center", va="center",
            fontsize=20, fontweight="bold", color="#1B3A6B")
    ax.text(0, -0.52, "/ 100", ha="center", va="center",
            fontsize=10, color="#7F8C8D")
 
    # Labels das faixas (posições fixas correspondendo às faixas)
    ax.text(-1.10, 0.20, "Crítico", ha="center", fontsize=7.5,
            color="#E74C3C", fontweight="bold")
    ax.text(0, 1.18, "Moderado", ha="center", fontsize=7.5,
            color="#F39C12", fontweight="bold")
    ax.text(1.10, 0.20, "Bom", ha="center", fontsize=7.5,
            color="#27AE60", fontweight="bold")
 
    plt.tight_layout(pad=0.5)
    buf = BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    buf.seek(0)
    return buf

## 4. GERAÇÃO DO PDF

In [73]:
def _estilo_tabela_base():
    return TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), COR_PRIMARIA),
        ("TEXTCOLOR",  (0, 0), (-1, 0), colors.white),
        ("FONTNAME",   (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE",   (0, 0), (-1, 0), 9),
        ("ALIGN",      (0, 0), (-1, 0), "CENTER"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, COR_NEUTRO]),
        ("FONTSIZE",   (0, 1), (-1, -1), 8),
        ("GRID",       (0, 0), (-1, -1), 0.4, colors.HexColor("#BDC3C7")),
        ("TOPPADDING", (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
    ])
 
 
def gerar_pdf(caminho_csv: str, df: pd.DataFrame, resultados: dict,
              scores: dict, score_geral: float, colunas_data: list,
              caminho_saida: str):
 
    doc = SimpleDocTemplate(
        caminho_saida,
        pagesize=A4,
        leftMargin=2 * cm,
        rightMargin=2 * cm,
        topMargin=2 * cm,
        bottomMargin=2 * cm,
    )
 
    estilos = getSampleStyleSheet()
    story = []
 
    # ── Estilos customizados ──
    titulo_style = ParagraphStyle("Titulo", parent=estilos["Title"],
        fontSize=20, textColor=COR_PRIMARIA, spaceAfter=4,
        fontName="Helvetica-Bold", alignment=TA_CENTER)
    subtitulo_style = ParagraphStyle("SubTitulo", parent=estilos["Normal"],
        fontSize=10, textColor=COR_SECUNDARIA, spaceAfter=2, alignment=TA_CENTER)
    h1_style = ParagraphStyle("H1", parent=estilos["Heading1"],
        fontSize=13, textColor=COR_PRIMARIA, spaceBefore=14, spaceAfter=4,
        fontName="Helvetica-Bold", borderPad=2)
    h2_style = ParagraphStyle("H2", parent=estilos["Heading2"],
        fontSize=10, textColor=COR_SECUNDARIA, spaceBefore=8, spaceAfter=3,
        fontName="Helvetica-Bold")
    normal_style = ParagraphStyle("Normal2", parent=estilos["Normal"],
        fontSize=8.5, textColor=COR_TEXTO, spaceAfter=2, leading=13)
    alerta_style = ParagraphStyle("Alerta", parent=estilos["Normal"],
        fontSize=8.5, textColor=COR_ALERTA, spaceAfter=2)
    ok_style = ParagraphStyle("OK", parent=estilos["Normal"],
        fontSize=8.5, textColor=COR_SUCESSO, spaceAfter=2)
 
    def secao(titulo, numero):
        story.append(Spacer(1, 0.3 * cm))
        story.append(HRFlowable(width="100%", thickness=1.5,
                                color=COR_PRIMARIA, spaceAfter=4))
        story.append(Paragraph(f"{numero}. {titulo}", h1_style))
 
    # ══ CAPA ══
    story.append(Spacer(1, 1.5 * cm))
    story.append(Paragraph("RELATÓRIO DE QUALIDADE DE DADOS", titulo_style))
    story.append(Paragraph("Avaliação pelos 6 Pilares de Qualidade", subtitulo_style))
    story.append(Spacer(1, 0.3 * cm))
    story.append(HRFlowable(width="100%", thickness=2, color=COR_DESTAQUE, spaceAfter=12))
 
    # Metadados
    meta = [
        ["Arquivo analisado", os.path.basename(caminho_csv)],
        ["Data do relatório", datetime.now().strftime("%d/%m/%Y %H:%M")],
        ["Total de registros", f"{len(df):,}".replace(",", ".")],
        ["Total de colunas", str(df.shape[1])],
        ["Colunas de data detectadas", ", ".join(colunas_data) if colunas_data else "Nenhuma"],
        ["Score Geral de Qualidade", f"{score_geral:.1f} / 100"],
    ]
    t = Table(meta, colWidths=[5 * cm, 11 * cm])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (0, -1), COR_NEUTRO),
        ("FONTNAME",   (0, 0), (0, -1), "Helvetica-Bold"),
        ("FONTSIZE",   (0, 0), (-1, -1), 8.5),
        ("GRID",       (0, 0), (-1, -1), 0.4, colors.HexColor("#BDC3C7")),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("LEFTPADDING", (0, 0), (-1, -1), 8),
        ("BACKGROUND", (0, 5), (-1, 5), COR_SECUNDARIA),
        ("TEXTCOLOR",  (0, 5), (-1, 5), colors.white),
        ("FONTNAME",   (0, 5), (-1, 5), "Helvetica-Bold"),
    ]))
    story.append(t)
    story.append(Spacer(1, 0.5 * cm))
 
    # Gauge + Radar lado a lado (radar menor para caber com a tabela na mesma pagina)
    buf_gauge = grafico_score_geral(score_geral)
    buf_radar = grafico_radar(scores)
    img_gauge = Image(buf_gauge, width=6.5 * cm, height=3.5 * cm)
    img_radar = Image(buf_radar, width=7.5 * cm, height=7.5 * cm)
 
    # Tabela de scores montada aqui para compor tudo junto
    dados_scores = [["Pilar", "Score", "Situação"]]
    scores_cores = []
    for pilar, sc in scores.items():
        if sc >= 80:
            sit, c = "Bom", COR_SUCESSO
        elif sc >= 60:
            sit, c = "Moderado", COR_DESTAQUE
        else:
            sit, c = "Crítico", COR_ALERTA
        dados_scores.append([pilar, f"{sc:.1f}", sit])
        scores_cores.append(c)
 
    t_scores = Table(dados_scores, colWidths=[5.5 * cm, 3 * cm, 3.5 * cm])
    ts = _estilo_tabela_base()
    for i, c in enumerate(scores_cores, start=1):
        ts.add("TEXTCOLOR", (1, i), (2, i), c)
        ts.add("FONTNAME", (1, i), (2, i), "Helvetica-Bold")
        ts.add("TOPPADDING", (0, i), (-1, i), 5)
        ts.add("BOTTOMPADDING", (0, i), (-1, i), 5)
    t_scores.setStyle(ts)
 
    # Coluna esquerda: gauge + titulo + tabela de scores
    from reportlab.platypus import KeepTogether
    col_esq = Table(
        [[img_gauge], [Paragraph("Resumo dos Pilares", h2_style)], [t_scores]],
        colWidths=[12 * cm]
    )
    col_esq.setStyle(TableStyle([
        ("ALIGN",  (0, 0), (-1, -1), "LEFT"),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("TOPPADDING",    (0, 0), (-1, -1), 2),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
    ]))
 
    # Layout principal: [esquerda | radar]
    layout = Table(
        [[col_esq, img_radar]],
        colWidths=[12 * cm, 9 * cm]
    )
    layout.setStyle(TableStyle([
        ("ALIGN",  (0, 0), (0, 0), "LEFT"),
        ("ALIGN",  (1, 0), (1, 0), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ]))
 
    story.append(KeepTogether(layout))
 
    story.append(PageBreak())
 
    # ══ PILAR 1 — COMPLETUDE ══
    secao("COMPLETUDE", 1)
    r = resultados["completude"]
    story.append(Paragraph(
        f"Score: <b>{r['score']:.1f}/100</b> &nbsp;|&nbsp; "
        f"Total de nulos: <b>{r['total_nulos']:,}</b> de <b>{r['total_celulas']:,}</b> células",
        normal_style))
 
    if r["alertas_obrigatorias"]:
        story.append(Paragraph("⚠ Alertas em colunas obrigatórias:", h2_style))
        for a in r["alertas_obrigatorias"]:
            story.append(Paragraph(f"• {a}", alerta_style))
 
    if r["colunas_criticas"]:
        story.append(Paragraph(f"Colunas abaixo de {CONFIG['completude_minima_pct']}% de preenchimento:", h2_style))
        dados_crit = [["Coluna", "% Preenchido"]]
        for col, pct in r["colunas_criticas"].items():
            dados_crit.append([col, f"{pct:.1f}%"])
        t = Table(dados_crit, colWidths=[12 * cm, 5 * cm])
        t.setStyle(_estilo_tabela_base())
        story.append(t)
    else:
        story.append(Paragraph("✔ Todas as colunas atingem o mínimo de preenchimento configurado.", ok_style))
 
    # Gráfico completude
    buf_comp = grafico_barras_completude(r["detalhe"])
    story.append(Spacer(1, 0.3 * cm))
    story.append(Image(buf_comp, width=16 * cm, height=max(6, len(r["detalhe"].head(15)) * 0.85) * cm))
 
    # ══ PILAR 2 — CONSISTÊNCIA ══
    secao("CONSISTÊNCIA", 2)
    r = resultados["consistencia"]
    story.append(Paragraph(
        f"Score: <b>{r['score']:.1f}/100</b> &nbsp;|&nbsp; "
        f"Duplicatas completas: <b>{r['duplicatas_completas']} ({r['pct_duplicatas']}%)</b>",
        normal_style))
 
    if r["duplicatas_colunas_unicas"]:
        story.append(Paragraph("Duplicatas em colunas identificadoras:", h2_style))
        dados_dup = [["Coluna", "Duplicatas"]]
        for col, n in r["duplicatas_colunas_unicas"].items():
            dados_dup.append([col, str(n)])
        t = Table(dados_dup, colWidths=[12 * cm, 5 * cm])
        t.setStyle(_estilo_tabela_base())
        story.append(t)
 
    if r["tipo_misto"]:
        story.append(Paragraph("Colunas com tipos misturados (texto + numérico):", h2_style))
        dados_tm = [["Coluna", "Qtd Numérico", "Qtd Texto"]]
        for col, info in r["tipo_misto"].items():
            dados_tm.append([col, str(info["numerico"]), str(info["texto"])])
        t = Table(dados_tm, colWidths=[9 * cm, 3.5 * cm, 3.5 * cm])
        t.setStyle(_estilo_tabela_base())
        story.append(t)
 
    if not r["duplicatas_colunas_unicas"] and not r["tipo_misto"] and r["duplicatas_completas"] == 0:
        story.append(Paragraph("✔ Nenhum problema de consistência detectado.", ok_style))
 
    # ══ PILAR 3 — CONFORMIDADE ══
    secao("CONFORMIDADE", 3)
    r = resultados["conformidade"]
    story.append(Paragraph(
        f"Score: <b>{r['score']:.1f}/100</b> &nbsp;|&nbsp; "
        f"Colunas de data analisadas: <b>{len(r['colunas_data_analisadas'])}</b> &nbsp;|&nbsp; "
        f"Datas inválidas: <b>{r['total_datas_invalidas']}</b>",
        normal_style))
 
    if r["datas_invalidas"]:
        dados_di = [["Coluna", "Datas Inválidas"]]
        for col, n in r["datas_invalidas"].items():
            dados_di.append([col, str(n)])
        t = Table(dados_di, colWidths=[12 * cm, 5 * cm])
        t.setStyle(_estilo_tabela_base())
        story.append(t)
    elif r["colunas_data_analisadas"]:
        story.append(Paragraph("✔ Todas as datas estão no formato esperado.", ok_style))
    else:
        story.append(Paragraph("ℹ Nenhuma coluna de data identificada nesta base.", normal_style))
 
    # ══ PILAR 4 — INTEGRIDADE ══
    secao("INTEGRIDADE", 4)
    r = resultados["integridade"]
    story.append(Paragraph(
        f"Score: <b>{r['score']:.1f}/100</b> &nbsp;|&nbsp; "
        f"Total de problemas encontrados: <b>{r['total_problemas']}</b>",
        normal_style))
 
    if r["valores_negativos_suspeitos"]:
        story.append(Paragraph("Valores negativos em campos que deveriam ser positivos:", h2_style))
        dados_neg = [["Coluna", "Qtd Negativos"]]
        for col, n in r["valores_negativos_suspeitos"].items():
            dados_neg.append([col, str(n)])
        t = Table(dados_neg, colWidths=[12 * cm, 5 * cm])
        t.setStyle(_estilo_tabela_base())
        story.append(t)
 
    if r["strings_vazias"]:
        story.append(Paragraph("Campos de texto com apenas espaços em branco:", h2_style))
        dados_sv = [["Coluna", "Qtd Registros"]]
        for col, n in r["strings_vazias"].items():
            dados_sv.append([col, str(n)])
        t = Table(dados_sv, colWidths=[12 * cm, 5 * cm])
        t.setStyle(_estilo_tabela_base())
        story.append(t)
 
    if r["total_problemas"] == 0:
        story.append(Paragraph("✔ Nenhum problema de integridade detectado.", ok_style))
 
    story.append(PageBreak())
 
    # ══ PILAR 5 — PRECISÃO ══
    secao("PRECISÃO", 5)
    r = resultados["precisao"]
    story.append(Paragraph(
        f"Score: <b>{r['score']:.1f}/100</b> &nbsp;|&nbsp; "
        f"Total de outliers: <b>{r['total_outliers']} ({r['pct_outliers']}%)</b>",
        normal_style))
 
    if r["outliers_por_coluna"]:
        dados_out = [["Coluna", "Outliers", "%", "Lim. Inferior", "Lim. Superior", "Min", "Max"]]
        for col, info in r["outliers_por_coluna"].items():
            dados_out.append([
                col,
                str(info["outliers"]),
                f"{info['pct']}%",
                str(info["limite_inf"]),
                str(info["limite_sup"]),
                str(info["min"]),
                str(info["max"]),
            ])
        t = Table(dados_out, colWidths=[4.5*cm, 2*cm, 1.8*cm, 2.5*cm, 2.5*cm, 2*cm, 2*cm])
        t.setStyle(_estilo_tabela_base())
        story.append(t)
        story.append(Paragraph(
            f"ℹ Método: IQR com fator {CONFIG['outlier_iqr_fator']}. "
            "Outliers não são necessariamente erros — avalie cada caso.",
            normal_style))
    else:
        story.append(Paragraph("✔ Nenhum outlier detectado nas colunas numéricas.", ok_style))
 
    # ══ PILAR 6 — ATUALIDADE ══
    secao("ATUALIDADE", 6)
    r = resultados["atualidade"]
    story.append(Paragraph(
        f"Score: <b>{r['score']:.1f}/100</b> &nbsp;|&nbsp; "
        f"Limite configurado: registros com mais de <b>{r['limite_dias']} dias</b> são sinalizados.",
        normal_style))
 
    if r["analise_por_coluna"]:
        dados_at = [["Coluna", "Registros", "Desatualizados", "%", "Mais Recente", "Mais Antigo"]]
        for col, info in r["analise_por_coluna"].items():
            dados_at.append([
                col,
                str(info["registros_analisados"]),
                str(info["desatualizados"]),
                f"{info['pct_desatualizados']}%",
                info["data_mais_recente"],
                info["data_mais_antiga"],
            ])
        t = Table(dados_at, colWidths=[3.5*cm, 2.2*cm, 2.8*cm, 1.8*cm, 3*cm, 3*cm])
        t.setStyle(_estilo_tabela_base())
        story.append(t)
    else:
        story.append(Paragraph("ℹ Nenhuma coluna de data identificada para avaliação de atualidade.", normal_style))
 
    # ══ RODAPÉ / CONCLUSÃO ══
    story.append(PageBreak())
    secao("CONCLUSÃO E RECOMENDAÇÕES", "")
 
    story.append(Paragraph(f"<b>Score Geral de Qualidade: {score_geral:.1f} / 100</b>", h1_style))
    story.append(Spacer(1, 0.3 * cm))
 
    recomendacoes = []
    r_comp = resultados["completude"]
    r_cons = resultados["consistencia"]
    r_conf = resultados["conformidade"]
    r_int  = resultados["integridade"]
    r_prec = resultados["precisao"]
    r_atu  = resultados["atualidade"]
 
    if r_comp["score"] < 80:
        recomendacoes.append(("Completude", f"Investigar as {len(r_comp['colunas_criticas'])} colunas com preenchimento abaixo do mínimo. \n Avaliar se o campo é realmente necessário ou se o processo de coleta pode ser melhorado."))
    if r_cons["duplicatas_completas"] > 0:
        recomendacoes.append(("Consistência", f"Remover ou investigar os {r_cons['duplicatas_completas']} registros duplicados antes de qualquer análise."))
    if r_conf["total_datas_invalidas"] > 0:
        recomendacoes.append(("Conformidade", f"Padronizar o formato de datas nas colunas com {r_conf['total_datas_invalidas']} registros inválidos. \nSugerir uso de formato ISO (YYYY-MM-DD) na origem."))
    if r_int["total_problemas"] > 0:
        recomendacoes.append(("Integridade", "Revisar regras de negócio na origem dos dados: \n campos obrigatórios, restrições de valores negativos e remoção de espaços em branco."))
    if r_prec["total_outliers"] > 0:
        recomendacoes.append(("Precisão", f"Validar manualmente os {r_prec['total_outliers']} outliers identificados. \n Confirmar se são erros de digitação ou valores legítimos extremos."))
    if r_atu["score"] < 80 and r_atu["analise_por_coluna"]:
        recomendacoes.append(("Atualidade", f"Verificar periodicidade de atualização dos dados. \n Registros mais antigos que {r_atu['limite_dias']} dias podem comprometer análises de tendências."))
 
    if recomendacoes:
        dados_rec = [["Pilar", "Recomendação"]]
        for pilar, rec in recomendacoes:
            dados_rec.append([pilar, rec])
        t = Table(dados_rec, colWidths=[3.5 * cm, 13 * cm])
        t.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), COR_PRIMARIA),
            ("TEXTCOLOR",  (0, 0), (-1, 0), colors.white),
            ("FONTNAME",   (0, 0), (-1, 0), "Helvetica-Bold"),
            ("FONTSIZE",   (0, 0), (-1, -1), 8.5),
            ("GRID",       (0, 0), (-1, -1), 0.4, colors.HexColor("#BDC3C7")),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, COR_NEUTRO]),
            ("TOPPADDING", (0, 0), (-1, -1), 5),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
            ("LEFTPADDING", (0, 0), (-1, -1), 6),
            ("VALIGN",     (0, 0), (-1, -1), "TOP"),
            ("FONTNAME",   (0, 1), (0, -1), "Helvetica-Bold"),
            ("TEXTCOLOR",  (0, 1), (0, -1), COR_SECUNDARIA),
        ]))
        story.append(t)
    else:
        story.append(Paragraph("✔ Base de dados com boa qualidade geral. Manter monitoramento periódico.", ok_style))
 
    story.append(Spacer(1, 0.5 * cm))
    story.append(HRFlowable(width="100%", thickness=1, color=COR_NEUTRO))
    story.append(Spacer(1, 0.2 * cm))
    story.append(Paragraph(
        f"Relatório gerado automaticamente em {datetime.now().strftime('%d/%m/%Y às %H:%M')} "
        f"| Sistema de Qualidade de Dados — 6 Pilares",
        ParagraphStyle("Rodape", parent=estilos["Normal"],
                       fontSize=7, textColor=colors.HexColor("#95A5A6"), alignment=TA_CENTER)))
 
    doc.build(story)
    print(f"\n✅ Relatório salvo em: {caminho_saida}")

## 5. EXECUÇÃO PRINCIPAL

In [74]:
def executar(caminho_csv: str, caminho_saida: str = None):
    print(f"\n📂 Carregando arquivo: {caminho_csv}")
    df = carregar_csv(caminho_csv)
    print(f"   ✔ {len(df):,} registros | {df.shape[1]} colunas")
 
    # Detecta colunas de data
    colunas_data = CONFIG["colunas_data"] or detectar_colunas_data(df)
    if colunas_data:
        print(f"   📅 Colunas de data detectadas: {colunas_data}")
 
    print("\n🔍 Avaliando os 6 pilares...")
    resultados = {
        "completude":   avaliar_completude(df),
        "consistencia": avaliar_consistencia(df),
        "conformidade": avaliar_conformidade(df, colunas_data),
        "integridade":  avaliar_integridade(df),
        "precisao":     avaliar_precisao(df),
        "atualidade":   avaliar_atualidade(df, colunas_data),
    }
 
    scores = {
        "Completude":  resultados["completude"]["score"],
        "Consistência": resultados["consistencia"]["score"],
        "Conformidade": resultados["conformidade"]["score"],
        "Integridade": resultados["integridade"]["score"],
        "Precisão":    resultados["precisao"]["score"],
        "Atualidade":  resultados["atualidade"]["score"],
    }
 
    score_geral = round(sum(scores.values()) / len(scores), 2)
 
    for pilar, sc in scores.items():
        status = "✅" if sc >= 80 else ("⚠️" if sc >= 60 else "❌")
        print(f"   {status} {pilar:<15} {sc:.1f}/100")
    print(f"\n   📊 SCORE GERAL: {score_geral:.1f}/100")
 
    if not caminho_saida:
        base = os.path.splitext(os.path.basename(caminho_csv))[0]
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        caminho_saida = os.path.join(os.path.dirname(caminho_csv), f"relatorio_qualidade_{base}_{ts}.pdf")
 
    print("\n📄 Gerando relatório PDF...")
    gerar_pdf(caminho_csv, df, resultados, scores, score_geral, colunas_data, caminho_saida)  # type: ignore[name-defined]
    return caminho_saida
 
 
# Detecta ambiente de execução
EM_JUPYTER = "ipykernel" in sys.modules
 
if not EM_JUPYTER:
    # Fora do Jupyter, aceita argumentos via linha de comando:
    #   python data_quality.py arquivo.csv [saida.pdf]
    if len(sys.argv) >= 2:
        ARQUIVO_CSV = sys.argv[1]
    if len(sys.argv) >= 3:
        ARQUIVO_PDF = sys.argv[2]
 
if not os.path.isfile(ARQUIVO_CSV):
    print(f"\n❌ Arquivo não encontrado: {ARQUIVO_CSV}")
    print("   Verifique o caminho na variável ARQUIVO_CSV no topo do bloco.")
else:
    executar(ARQUIVO_CSV, ARQUIVO_PDF)


📂 Carregando arquivo: ../dataset/Cleaned_Laptop_data.csv
   ✔ 896 registros | 23 colunas

🔍 Avaliando os 6 pilares...
   ✅ Completude      100.0/100
   ✅ Consistência    93.9/100
   ✅ Conformidade    100.0/100
   ✅ Integridade     100.0/100
   ✅ Precisão        87.0/100
   ✅ Atualidade      100.0/100

   📊 SCORE GERAL: 96.8/100

📄 Gerando relatório PDF...

✅ Relatório salvo em: ../dataset/relatorio_qualidade_Cleaned_Laptop_data_20260314_210454.pdf
